Q1: What is the total withdrawn IPO value (in $ millions) for the company class with the highest total withdrawal value?

In [1]:
import pandas as pd
import requests as req

In [2]:
url =f"https://www.iposcoop.com/ipos-recently-filed/"
headers = {
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/58.0.3029.110 Safari/537.3'
        )
    }
from io import StringIO

def getdata(url):
    try:
            response = req.get(url, headers=headers, timeout=10)
            response.raise_for_status()

            # Wrap HTML text in StringIO to avoid deprecation warning
            # "Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object."
            html_io = StringIO(response.text)
            tables = pd.read_html(html_io)

            if not tables:
                raise ValueError(f"No tables found.")

            return tables[0]

    except req.exceptions.RequestException as e:
            print(f"Request failed: {e}")
    except ValueError as ve:
            print(f"Data error: {ve}")
    except Exception as ex:
            print(f"Unexpected error: {ex}")

    return pd.DataFrame()


In [3]:
df=getdata(url)

In [4]:
df.columns=df.columns.str.replace(" ","_").str.lower()

In [5]:
from datetime import datetime
df_filtered=df[(df["expected_to_trade"]=="Withdrawn") & ((pd.to_datetime(df['file_date'], format='mixed')) <datetime(2026,9,11))]

In [6]:
df_filtered.head(1)

,file_date,company,symbol,managers,shares_(millions),price_low,price_high,est_$_vol_(millions),expected_to_trade,scoop_rating
38,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.0,NaN,NaN,$100.00,Withdrawn,S/O


In [7]:
from numpy import append
words={"Technologies":"Technologies"
,"Acquisition Corp":"Acquisition Corp"
, "Acquisition Corporation":"Acquisition Corp"
,"Corp":"Acquisition Corp"
,"Inc":"Inc."
,"Incorporated":"Inc."
,"Group":"Group"
,"Ltd":"Limited"
,"Limited":"Limited"
,"Holdings":"Holdings" 
,"Holding":"Holdings"
,"Others":"Others"
}


company_type_data=[]
import re
for comp in df_filtered["company"]:
    txt = comp
    wordarray=[]
    for w in words.keys():
        current_pos = re.search(w, txt)
        if current_pos:
            wordarray.append(w)
        
    if wordarray:
            company_type_data.append(words[wordarray[0]])
    else:
        company_type_data.append("N/A")

df_filtered["company_type"]=company_type_data

In [8]:
df_filtered.head(1)

,file_date,company,symbol,managers,shares_(millions),price_low,price_high,est_$_vol_(millions),expected_to_trade,scoop_rating,company_type
38,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.0,NaN,NaN,$100.00,Withdrawn,S/O,Technologies


In [9]:
# df_filtered.dtypes
df_filtered["avg_price"]=(pd.to_numeric(df_filtered["price_high"].str.replace("$","").fillna(0),errors='coerce')+pd.to_numeric(df_filtered["price_low"].str.replace("$","").fillna(0),errors='coerce'))/2

In [10]:
df_filtered

,file_date,company,symbol,managers,shares_(millions),price_low,price_high,est_$_vol_(millions),expected_to_trade,scoop_rating,company_type,avg_price
38,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.00,NaN,NaN,$100.00,Withdrawn,S/O,Technologies,0.00
43,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.00,$4.00,$5.00,$9.00,Withdrawn,S/O,Holdings,4.50
50,2026-09-01,Hornbeck Offshore Services (Withdrawn),HOS,J.P. Morgan/ Barclays/DNB Markets/Piper Sandle...,0.00,NaN,NaN,$100.00,Withdrawn,S/O,N/A,0.00
52,2026-08-31,Coolbit Technologies Ltd. (Withdrawn),CBAI,Eddid Securities USA,5.00,$4.00,$5.00,$22.50,Withdrawn,S/O,Technologies,4.50
59,2026-08-27,Timber Road Acquisition Corp. (Withdrawn),TMRDU,Roth Capital Partners/Stone X Financial Inc.,20.00,$10.00,$10.00,$200.00,Withdrawn,S/O,Acquisition Corp,10.00
69,2026-08-24,pan-Africa Corp. (Withdrawn),BLEUU,Santander,12.50,$10.00,$10.00,$125.00,Withdrawn,S/O,Acquisition Corp,10.00
84,2026-08-18,Living Homeopathy (Withdrawn),LHI,Revere Securities,3.75,$4.00,$6.00,$19.00,Withdrawn,S/O,N/A,5.00
128,2026-07-31,Nuclea Energy (Withdrawn),NCLA,Joseph Gunnar & Co.,5.60,$8.00,$10.00,$50.40,Withdrawn,S/O,N/A,9.00
158,2026-07-10,APEX Global Solutions Ltd. (WITHDRAWN),APEX,US Tiger Securities,3.75,$4.00,$4.00,$15.00,Withdrawn,S/O,Limited,4.00
163,2026-07-08,Cloud Data Holdings (Withdrawn),CDN,Cathay Securities,3.75,$4.00,$4.50,$15.94,Withdrawn,S/O,Holdings,4.25


In [11]:
df_filtered.info()


<class 'pandas.DataFrame'>
Index: 31 entries, 38 to 491
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   file_date             31 non-null     str    
 1   company               31 non-null     str    
 2   symbol                31 non-null     str    
 3   managers              31 non-null     str    
 4   shares_(millions)     31 non-null     float64
 5   price_low             25 non-null     str    
 6   price_high            25 non-null     str    
 7   est_$_vol_(millions)  31 non-null     str    
 8   expected_to_trade     31 non-null     str    
 9   scoop_rating          31 non-null     str    
 10  company_type          31 non-null     str    
 11  avg_price             31 non-null     float64
dtypes: float64(2), str(10)
memory usage: 6.7 KB


In [12]:
df_filtered["est_$_vol_(millions)"]=pd.to_numeric(df_filtered["est_$_vol_(millions)"].str.replace("$","").str.replace(",",""),errors='coerce')


In [13]:
df_filtered.info()

<class 'pandas.DataFrame'>
Index: 31 entries, 38 to 491
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   file_date             31 non-null     str    
 1   company               31 non-null     str    
 2   symbol                31 non-null     str    
 3   managers              31 non-null     str    
 4   shares_(millions)     31 non-null     float64
 5   price_low             25 non-null     str    
 6   price_high            25 non-null     str    
 7   est_$_vol_(millions)  31 non-null     float64
 8   expected_to_trade     31 non-null     str    
 9   scoop_rating          31 non-null     str    
 10  company_type          31 non-null     str    
 11  avg_price             31 non-null     float64
dtypes: float64(3), str(9)
memory usage: 6.5 KB


In [14]:
df_filtered.head(1)

,file_date,company,symbol,managers,shares_(millions),price_low,price_high,est_$_vol_(millions),expected_to_trade,scoop_rating,company_type,avg_price
38,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.0,NaN,NaN,100.0,Withdrawn,S/O,Technologies,0.0


In [15]:
import numpy as np

In [16]:
df_filtered["Shares_offered_value"] = np.where(
    pd.notna(df_filtered["shares_(millions)"] * df_filtered["avg_price"]),
    df_filtered["shares_(millions)"] * df_filtered["avg_price"],
    df_filtered["est_$_vol_(millions)"]
)


In [17]:
df_filtered

,file_date,company,symbol,managers,shares_(millions),price_low,price_high,est_$_vol_(millions),expected_to_trade,scoop_rating,company_type,avg_price,Shares_offered_value
38,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.00,NaN,NaN,100.00,Withdrawn,S/O,Technologies,0.00,0.0000
43,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.00,$4.00,$5.00,9.00,Withdrawn,S/O,Holdings,4.50,9.0000
50,2026-09-01,Hornbeck Offshore Services (Withdrawn),HOS,J.P. Morgan/ Barclays/DNB Markets/Piper Sandle...,0.00,NaN,NaN,100.00,Withdrawn,S/O,N/A,0.00,0.0000
52,2026-08-31,Coolbit Technologies Ltd. (Withdrawn),CBAI,Eddid Securities USA,5.00,$4.00,$5.00,22.50,Withdrawn,S/O,Technologies,4.50,22.5000
59,2026-08-27,Timber Road Acquisition Corp. (Withdrawn),TMRDU,Roth Capital Partners/Stone X Financial Inc.,20.00,$10.00,$10.00,200.00,Withdrawn,S/O,Acquisition Corp,10.00,200.0000
69,2026-08-24,pan-Africa Corp. (Withdrawn),BLEUU,Santander,12.50,$10.00,$10.00,125.00,Withdrawn,S/O,Acquisition Corp,10.00,125.0000
84,2026-08-18,Living Homeopathy (Withdrawn),LHI,Revere Securities,3.75,$4.00,$6.00,19.00,Withdrawn,S/O,N/A,5.00,18.7500
128,2026-07-31,Nuclea Energy (Withdrawn),NCLA,Joseph Gunnar & Co.,5.60,$8.00,$10.00,50.40,Withdrawn,S/O,N/A,9.00,50.4000
158,2026-07-10,APEX Global Solutions Ltd. (WITHDRAWN),APEX,US Tiger Securities,3.75,$4.00,$4.00,15.00,Withdrawn,S/O,Limited,4.00,15.0000
163,2026-07-08,Cloud Data Holdings (Withdrawn),CDN,Cathay Securities,3.75,$4.00,$4.50,15.94,Withdrawn,S/O,Holdings,4.25,15.9375


In [27]:
# Answer

df_filtered.groupby("company_type").agg(
    Shares_offered_value_sum=("Shares_offered_value", "sum")
).sort_values("Shares_offered_value_sum", ascending=False).head(1)

,Shares_offered_value_sum
company_type,
Acquisition Corp,499.985
